In [1]:
import pandas as pd
import os
import cv2
import albumentations as A
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from tqdm import tqdm
import glob
import warnings
import csv
from collections import OrderedDict
from sklearn.model_selection import train_test_split
import os, shutil, pathlib
import random
from tensorflow.keras.utils import load_img,img_to_array

2026-05-23 14:40:28.915477: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-23 14:40:28.941806: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-05-23 14:40:29.464493: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
import tensorflow as tf
from tensorflow import keras

In [3]:
import tensorflow as tf
import torch

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    tf.config.set_logical_device_configuration(
        gpus[0], [tf.config.LogicalDeviceConfiguration(memory_limit=4000)])
    print("VRAM Partagée : 4Go TF / 4Go Torch")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

VRAM Partagée : 4Go TF / 4Go Torch


In [4]:
print("GPU détectés par TF :", tf.config.list_physical_devices('GPU'))

GPU détectés par TF : [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [5]:
####### CBIS DDSM #######

In [6]:
BASE_INPUT_PATH = "/home/bigbro/Dataset_CBIS"
CSV_FOLDER_PATH = os.path.join(BASE_INPUT_PATH, "csv")
IMAGE_FOLDER_PATH = os.path.join(BASE_INPUT_PATH, "jpeg")
BASE_OUTPUT_PATH = "/home/bigbro/"

In [7]:
IMAGE_SIZE = 256
BATCH_SIZE = 16
VALIDATION_SPLIT = 0.2
LEARNING_RATE = 1e-4
NUM_EPOCHS = 100
RANDOM_SEED = 42

In [8]:
def find_image_in_folder(folder_path):
    """
    Trouve la première image .jpg ou .png dans le dossier ET ses sous-dossiers.
    """
    if not folder_path or not os.path.isdir(folder_path):
        return None
        
    for root, dirs, files in os.walk(folder_path):
        for file in files:
            if file.lower().endswith(('.jpg', '.png')):
                return os.path.join(root, file)
                
    return None

def compute_all_bounding_boxes(mask_path, min_area=100):
    """
    Returns a list of bounding boxes [[x_min, y_min, width, height],...]
    Returns None if mask doesn't exist or is invalid
    """
    if not os.path.exists(mask_path):
        return None

    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    if mask is None:
        return None

    _, thresh = cv2.threshold(mask, 1, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    boxes = []
    for contour in contours:
        x, y, w, h = cv2.boundingRect(contour)
        if w * h < min_area:
            continue
        boxes.append([x, y, w, h])

    return boxes if boxes else None

def build_metadata_lookup(dicom_info_path, jpeg_base_dir, *args):
    print(f"Building metadata lookup from: {dicom_info_path}")
    master_map = {}
    
    try:
        dicom_info = pd.read_csv(dicom_info_path, dtype=str)
    except FileNotFoundError:
        print(f"Error: Metadata file not found at {dicom_info_path}")
        return master_map
        
    valid_descriptions = {arg for arg in args}
    filtered_df = dicom_info[dicom_info['SeriesDescription'].isin(valid_descriptions)]

    for _, row in tqdm(filtered_df.iterrows(), total=len(filtered_df), desc="Building lookup map"):
        series_desc = row['SeriesDescription'] 
        patient_id_composite = row['PatientID'] 
        
        full_path = None
        
        if 'image_path' in row and pd.notna(row['image_path']):
            rel_path = row['image_path']
            if 'jpeg' in rel_path:
                clean_rel_path = rel_path.split('jpeg')[-1].strip("/\\")
                tmp_path = os.path.join(jpeg_base_dir, clean_rel_path)
            else:
                tmp_path = os.path.join(jpeg_base_dir, rel_path)

            if os.path.exists(tmp_path):
                full_path = tmp_path

        if not full_path:
            series_uid = row['SeriesInstanceUID']
            folder_path = os.path.join(jpeg_base_dir, series_uid)
            full_path = find_image_in_folder(folder_path)
            
        if not full_path or not os.path.exists(full_path): 
            continue

        if patient_id_composite not in master_map:
            master_map[patient_id_composite] = {}
            
        master_map[patient_id_composite][series_desc] = full_path
            
    print(f"Metadata lookup map built. Found {len(master_map)} unique composite keys.")
    return master_map

In [9]:
def BuildMasterDataset(MASTER_LIST_PATH="/home/bigbro/Dataset_CBIS/master_dataset.csv", 
                       argument1="cropped images", 
                       argument2="ROI mask images"):
    """
    Build master dataset. If ROI mask is missing:
    - For BENIGN cases: sets mask path to 'n/a' and bbox to 'n/a'
    - For MALIGNANT cases: skips the image entirely
    """
    BASE_INPUT_PATH = "/home/bigbro/Dataset_CBIS"
    IMAGE_FOLDER_PATH = os.path.join(BASE_INPUT_PATH, "jpeg")
    DICOM_INFO_PATH = os.path.join(BASE_INPUT_PATH, "csv/dicom_info.csv")
    
    INPUT_CSVS = [
        os.path.join(BASE_INPUT_PATH, "csv/mass_case_description_train_set.csv"),
        os.path.join(BASE_INPUT_PATH, "csv/mass_case_description_test_set.csv"),
        os.path.join(BASE_INPUT_PATH, "csv/calc_case_description_train_set.csv"),
        os.path.join(BASE_INPUT_PATH, "csv/calc_case_description_test_set.csv")
    ]

    # Build the authoritative map
    master_map = build_metadata_lookup(DICOM_INFO_PATH, IMAGE_FOLDER_PATH, argument1, argument2)
    
    if not master_map:
        return

    found_pairs_count = 0
    missing_mask_count = 0
    skipped_malignant_count = 0
    
    with open(MASTER_LIST_PATH, 'w', newline='') as outfile:
        csv_writer = csv.writer(outfile)
        csv_writer.writerow([
            'cropped_image_path', 'roi_mask_path',
            'x_min', 'y_min', 'width', 'height',
            'pathology', 'assessment', 'patient_id', 'series_type', 'mask_status'
        ])

        for filepath in INPUT_CSVS:
            filename = os.path.basename(filepath)
            if not filepath or not os.path.exists(filepath):
                continue
            
            # Determine prefixes
            if "mass" in filename.lower():
                type_prefix = "Mass"
            elif "calc" in filename.lower():
                type_prefix = "Calc"
            else:
                continue
                
            if "train" in filename.lower():
                split_prefix = "Training"
            elif "test" in filename.lower():
                split_prefix = "Test"
            else:
                continue
                
            full_prefix = f"{type_prefix}-{split_prefix}"

            with open(filepath, "r") as infile:
                csv_reader = csv.reader(infile)
                header = next(csv_reader)
                
                pathology_idx = header.index('pathology')
                assessment_idx = header.index('assessment')
                patient_id_idx = header.index('patient_id')
                breast_idx = header.index('left or right breast')
                view_idx = header.index('image view')
                abnormality_id_idx = header.index('abnormality id')

                for row in tqdm(csv_reader, desc=f"Processing {filename}"):
                    if not any(row):
                        continue
                    
                    pathology = row[pathology_idx]
                    assessment = row[assessment_idx]
                    patient_id = row[patient_id_idx]
                    side = row[breast_idx]
                    view = row[view_idx]
                    abn_id = row[abnormality_id_idx]
                    
                    try:
                        abn_id_clean = str(int(float(abn_id)))
                    except ValueError:
                        abn_id_clean = str(abn_id).strip()

                    composite_key = f"{full_prefix}_{patient_id}_{side}_{view}_{abn_id_clean}"
                    
                    study_data = master_map.get(composite_key)
                    if not study_data:
                        continue

                    full_crop_path = study_data.get('cropped images')
                    full_mask_path = study_data.get('ROI mask images')
                    
                    # Check if cropped image exists
                    if not full_crop_path:
                        continue
                    
                    mask_status = 'valid'
                    
                    # Handle missing mask based on pathology
                    if not full_mask_path:
                        # Check if pathology is BENIGN or BENIGN_WITHOUT_CALLBACK
                        pathology_upper = str(pathology).upper()
                        is_benign = 'BENIGN' in pathology_upper and 'MALIGNANT' not in pathology_upper
                        
                        if is_benign:
                            # Set mask path to 'n/a' for benign cases without mask
                            full_mask_path = 'n/a'
                            mask_status = 'n/a'
                            missing_mask_count += 1
                            
                            # Write entry with n/a values
                            csv_writer.writerow([
                                full_crop_path,
                                'n/a',  # roi_mask_path
                                'n/a', 'n/a', 'n/a', 'n/a',  # bbox coordinates
                                pathology,
                                assessment,
                                patient_id,
                                full_prefix,
                                'n/a'  # mask_status
                            ])
                            found_pairs_count += 1
                        else:
                            # Skip malignant cases without masks
                            skipped_malignant_count += 1
                        continue
                    
                    # Safety check: if paths are identical, skip
                    if full_crop_path == full_mask_path and mask_status == 'valid':
                        continue
                    
                    # Compute bounding boxes
                    boxes = compute_all_bounding_boxes(full_mask_path, min_area=100)
                     
                    # If mask is valid but no boxes found
                    if boxes is None:
                        # Write entry with n/a values for bounding box
                        csv_writer.writerow([
                            full_crop_path,
                            full_mask_path,
                            'n/a', 'n/a', 'n/a', 'n/a',
                            pathology,
                            assessment,
                            patient_id,
                            full_prefix,
                            mask_status
                        ])
                        found_pairs_count += 1
                    else:
                        # Write entry for each bounding box
                        for (x_min, y_min, width, height) in boxes:
                            csv_writer.writerow([
                                full_crop_path,
                                full_mask_path,
                                x_min, y_min, width, height,
                                pathology,
                                assessment,
                                patient_id,
                                full_prefix,
                                mask_status
                            ])
                            found_pairs_count += 1

    print(f"\n{'='*60}")
    print(f"DATASET BUILD SUMMARY")
    print(f"{'='*60}")
    print(f"Master list saved to: {MASTER_LIST_PATH}")
    print(f"Valid pairs found: {found_pairs_count}")
    print(f"Benign cases without masks (n/a): {missing_mask_count}")
    print(f"Malignant cases skipped (no mask): {skipped_malignant_count}")
    print(f"{'='*60}")

BuildMasterDataset()

Building metadata lookup from: /home/bigbro/Dataset_CBIS/csv/dicom_info.csv


Building lookup map: 100%|████████████████| 6814/6814 [00:00<00:00, 8463.84it/s]


Metadata lookup map built. Found 3539 unique composite keys.


Processing mass_case_description_train_set.csv: 1318it [00:19, 68.71it/s]
Processing mass_case_description_test_set.csv: 378it [00:05, 69.60it/s]
Processing calc_case_description_train_set.csv: 1546it [00:22, 68.99it/s]
Processing calc_case_description_test_set.csv: 326it [00:00, 3263.48it/s]


DATASET BUILD SUMMARY
Master list saved to: /home/bigbro/Dataset_CBIS/master_dataset.csv
Valid pairs found: 3416
Benign cases without masks (n/a): 192
Malignant cases skipped (no mask): 126


In [9]:
MASTER_LIST_PATH = "/home/bigbro/Dataset_CBIS/master_dataset.csv"

df_master = pd.read_csv(MASTER_LIST_PATH, keep_default_na=False)

unique_patients = df_master["patient_id"].unique()
train_patients, val_patients = train_test_split(
    unique_patients,
    test_size=VALIDATION_SPLIT,
    random_state=RANDOM_SEED
)

print(f"Splitting {len(unique_patients)} unique patients: {len(train_patients)} for training, {len(val_patients)} for validation.")

Splitting 1508 unique patients: 1206 for training, 302 for validation.


In [10]:
train_df = df_master[df_master['patient_id'].isin(train_patients)].reset_index(drop=True)
val_df = df_master[df_master['patient_id'].isin(val_patients)].reset_index(drop=True)
train_transforms = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.3),
    A.Rotate(limit=20, p=0.5),
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.2, rotate_limit=20, p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.5),
    A.ElasticTransform(alpha=1, sigma=50, alpha_affine=50, p=0.3),
    A.GridDistortion(p=0.3),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
   
])

val_transforms = A.Compose([
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    
])

/home/bigbro/miniconda3/lib/python3.13/site-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)
/tmp/ipykernel_14071/819158041.py:9: UserWarning: Argument(s) 'alpha_affine' are not valid for transform ElasticTransform
  A.ElasticTransform(alpha=1, sigma=50, alpha_affine=50, p=0.3),


In [11]:
val_df = val_df.sample(frac=1, random_state=42).reset_index(drop=True)

In [12]:
normal_cases_indices = val_df[val_df['mask_status'] == 'n/a'].index.tolist()

print(f"Indices disponibles pour les cas normaux : {normal_cases_indices[:10]}") 

Indices disponibles pour les cas normaux : [21, 59, 64, 69, 77, 109, 115, 130, 152, 159]


In [13]:
class MultimodalSequence(tf.keras.utils.Sequence):
    def __init__(self, dataframe, clinical_dict, batch_size=16, img_size=256, transforms=None):
        self.df = dataframe
        self.clinical_dict = clinical_dict
        self.batch_size = batch_size
        self.img_size = img_size
        self.transforms = transforms

    def __len__(self):
        return int(np.ceil(len(self.df) / self.batch_size))

    def __getitem__(self, idx):
        batch_df = self.df.iloc[idx * self.batch_size : (idx + 1) * self.batch_size]
        
        imgs, msks, embs = [], [], []

        for _, row in batch_df.iterrows():
            image, mask = self._load_data(row)
            
            p_id = str(row['patient_id'])
            embedding = self.clinical_dict.get(p_id, np.zeros(768))
            
            imgs.append(image)
            msks.append(mask)
            embs.append(embedding)

        return (np.array(imgs), np.array(embs)), np.array(msks)

    def _load_data(self, row):
        image_path = row["cropped_image_path"]
        image = cv2.imread(image_path)
        if image is None:
            image = np.zeros((self.img_size, self.img_size, 3), dtype=np.uint8)
        else:
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            image = cv2.resize(image, (self.img_size, self.img_size))

        mask_path = row["roi_mask_path"]
        if row["mask_status"] == 'n/a' or mask_path == 'n/a':
            mask = np.zeros((self.img_size, self.img_size), dtype=np.uint8)
        else:
            full_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
            if full_mask is None:
                mask = np.zeros((self.img_size, self.img_size), dtype=np.uint8)
            else:
                x_min = max(0, int(float(row['x_min'])))
                y_min = max(0, int(float(row['y_min'])))
                x_max = min(full_mask.shape[1], x_min + int(float(row['width'])))
                y_max = min(full_mask.shape[0], y_min + int(float(row['height'])))
                
                mask_crop = full_mask[y_min:y_max, x_min:x_max]
                
                if mask_crop.size == 0:
                    mask = np.zeros((self.img_size, self.img_size), dtype=np.uint8)
                else:
                    mask = cv2.resize(mask_crop, (self.img_size, self.img_size))

        _, mask = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY)
        mask = mask.astype(np.float32) / 255.0

        if self.transforms:
            transformed = self.transforms(image=image, mask=mask)
            image = transformed["image"]
            mask = transformed["mask"]

        if mask.ndim == 2:
            mask = np.expand_dims(mask, axis=-1)
            
        return image, mask

In [14]:

import pickle
with open("/home/bigbro/Dataset_CBIS/clinical_embeddings_dict_v2.pkl", "rb") as f:
    clinical_dict = pickle.load(f)

train_generator = MultimodalSequence(
    dataframe=train_df, 
    clinical_dict=clinical_dict,
    batch_size=BATCH_SIZE, 
    img_size=IMAGE_SIZE,
    transforms=train_transforms
)

val_generator = MultimodalSequence(
    dataframe=val_df, 
    clinical_dict=clinical_dict,
    batch_size=BATCH_SIZE, 
    img_size=IMAGE_SIZE,
    transforms=val_transforms
)

print(f" Générateurs Multimodaux prêts !")
print(f"Échantillons : Train={len(train_df)} | Val={len(val_df)}")

 Générateurs Multimodaux prêts !
Échantillons : Train=2731 | Val=685


In [15]:
base_dir=pathlib.Path("archive/Dataset_BUSI_with_GT") #Source doc
output_dir=pathlib.Path("archive/BUSI_Splitted")

In [16]:
base_path = pathlib.Path("archive/BUSI_Splitted")
input_img_paths = sorted([str(p) for p in (base_path).rglob("inputs/*/*.png")])
target_paths = sorted([str(p) for p in (base_path).rglob("targets/*/*.png")])
print(f"Total images trouvées : {len(input_img_paths)}")
print(f"Total masques trouvés : {len(target_paths)}")

Total images trouvées : 780
Total masques trouvés : 780


In [17]:
img_size=(256,256)
num_imgs=len(input_img_paths)
random.Random(1337).shuffle(input_img_paths)
random.Random(1337).shuffle(target_paths)

def path_to_input_image(path):
    return img_to_array(load_img(path, target_size=img_size))
def path_to_target(path):
    img = img_to_array(
        load_img(path, target_size=img_size, color_mode="grayscale"))
    img=(img>127).astype("uint8")
    return img 

input_imgs=np.zeros((num_imgs,)+img_size+(3,),dtype="float32")
targets=np.zeros((num_imgs,)+img_size+(1,), dtype="uint8")
for i in range(num_imgs):
    input_imgs[i]=path_to_input_image(input_img_paths[i])
    targets[i]=path_to_target(target_paths[i])
num_val_samples=200
train_input_imgs=input_imgs[:-num_val_samples]
train_targets=targets[:-num_val_samples]
val_input_imgs=input_imgs[-num_val_samples:]
val_targets=targets[-num_val_samples:]

In [18]:
from tensorflow.keras import backend as K
def dice_coef(y_true,y_pred,smooth=1e-6):
    # we flat out tensor
    y_true_f=K.flatten(K.cast(y_true,'float32'))
    y_pred_f=K.flatten(y_pred)

    #calculate intersection
    intersection=K.sum(y_true_f*y_pred_f)
    return (2.*intersection+smooth) / (K.sum(y_true_f)+K.sum(y_pred_f)+smooth)
def dice_loss(y_true,y_pred):
    return 1-dice_coef(y_true,y_pred)

In [19]:
def specificity(y_true, y_pred):
    y_true = K.cast(y_true, 'float32')
    true_negatives = K.sum(K.round(K.clip((1 - y_true) * (1 - y_pred), 0, 1)))
    possible_negatives = K.sum(K.round(K.clip(1 - y_true, 0, 1)))
    return true_negatives / (possible_negatives + K.epsilon())

def f1_score(y_true, y_pred):
    p = keras.metrics.Precision()(y_true, y_pred)
    r = keras.metrics.Recall()(y_true, y_pred)
    return 2 * ((p * r) / (p + r + K.epsilon()))

In [20]:
def log_cosh_dice_loss(y_true, y_pred):
    x = dice_loss(y_true, y_pred)
    return tf.math.log(tf.math.cosh(x))

In [21]:
def bce_dice_loss(y_true, y_pred):
    # BCE 
    bce = tf.keras.losses.binary_crossentropy(y_true, y_pred)
    # Dice Loss
    dice_loss = 1.0 - dice_coef(y_true, y_pred)
    return bce + dice_loss

In [22]:
def pure_dice_loss(y_true, y_pred):
    return dice_loss(y_true, y_pred)

In [23]:
custom_dict = {
    "bce_dice_loss": bce_dice_loss,
    "pure_dice_loss": pure_dice_loss,
    "log_cosh_dice_loss": log_cosh_dice_loss,
    "dice_loss": dice_loss,
    "dice_coef": dice_coef,
    "specificity": specificity,            
}

In [24]:
from transformers import pipeline
import numpy as np
import pickle

pipe = pipeline("feature-extraction", 
                model="emilyalsentzer/Bio_ClinicalBERT", 
                framework="tf",
                device=-1)

def get_embedding_with_pipe(text):
    features = pipe(text)
    feat_array = np.array(features[0])
    return np.mean(feat_array, axis=0)

print("Lecture du CSV sur Kaggle...")
df_meta = pd.read_csv("/Models/mass_case_description_train_set.csv")


def combine_features(row):
    # Cas normal
    if row.get('mask_status') == 'n/a' or pd.isna(row.get('pathology')):
        return "Normal breast tissue, no mass or calcification detected."
    
    return f"A {row['pathology']} mass with {row['mass shape']} shape and {row['mass margins']} margins."

print("Préparation des phrases...")
text_descriptions = df_meta.apply(combine_features, axis=1).tolist()

print(f"Génération des embeddings pour {len(text_descriptions)} lignes...")

clinical_map = {}
print(" Génération des embeddings robustes (Mean Pooling)...")

for _, row in tqdm(df_meta.iterrows(), total=len(df_meta)):
    text = combine_features(row)
    vec = get_embedding_with_pipe(text)
    
    p_id = str(row['patient_id'])
    clinical_map[p_id] = vec

with open("clinical_embeddings_dict_v2.pkl", "wb") as f:
    pickle.dump(clinical_map, f)

print(f"Dictionnaire enregistré ! {len(clinical_map)} patients traités.")

pytorch_model.bin:   0%|          | 0.00/436M [00:00<?, ?B/s]

Cancellation requested; stopping current tasks.


KeyboardInterrupt: 

In [25]:
import torch.nn as nn
class CBAM(nn.Module):
    def __init__(self, channels):
        super(CBAM, self).__init__()
        # Channel Attention
        self.ca = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(channels, channels // 8, 1, bias=False),
            nn.ReLU(),
            nn.Conv2d(channels // 8, channels, 1, bias=False),
            nn.Sigmoid()
        )
        # Spatial Attention
        self.sa = nn.Sequential(
            nn.Conv2d(2, 1, 7, padding=3, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        x = x * self.ca(x)
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        spatial = self.sa(torch.cat([avg_out, max_out], dim=1))
        return x * spatial


import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
from transformers import CLIPSegProcessor, CLIPSegForImageSegmentation
class VGG19_CLIP_Hybrid_SOTA(nn.Module):
    def __init__(self, frozen_clipseg):
        super(VGG19_CLIP_Hybrid_SOTA, self).__init__()
        self.clipseg = frozen_clipseg
        for param in self.clipseg.parameters(): param.requires_grad = False
        
        vgg = models.vgg19(pretrained=True).features
        self.enc1, self.enc2 = vgg[:4], vgg[4:9]
        self.enc3, self.enc4 = vgg[9:18], vgg[18:27]
        self.enc5 = vgg[27:36]
        
        for p in self.enc1.parameters(): p.requires_grad = False
        for p in self.enc2.parameters(): p.requires_grad = False

        self.cbam5 = CBAM(512)
        self.cbam4 = CBAM(512)
        self.cbam3 = CBAM(256)
        self.cbam2 = CBAM(128)
        self.cbam1 = CBAM(64)

        self.up4 = nn.ConvTranspose2d(512 + 1, 512, 2, 2) 
        self.dec4 = nn.Sequential(
            nn.Conv2d(1024, 512, 3, padding=1), nn.ReLU(),
            nn.Conv2d(512, 512, 3, padding=1), nn.ReLU(),
            nn.Dropout(0.2)
        )
        
        self.up3 = nn.ConvTranspose2d(512, 256, 2, 2)
        self.dec3 = nn.Sequential(
            nn.Conv2d(512, 256, 3, padding=1), nn.ReLU(),
            nn.Conv2d(256, 256, 3, padding=1), nn.ReLU()
        )
        
        # ... même logique pour up2 et up1 ...
        self.up2 = nn.ConvTranspose2d(256, 128, 2, 2)
        self.dec2 = nn.Sequential(nn.Conv2d(256, 128, 3, padding=1), nn.ReLU(), nn.Conv2d(128, 128, 3, padding=1), nn.ReLU())
        
        self.up1 = nn.ConvTranspose2d(128, 64, 2, 2)
        self.dec1 = nn.Sequential(nn.Conv2d(128, 64, 3, padding=1), nn.ReLU(), 
                                  nn.Conv2d(64, 1, 1), nn.Sigmoid())

    def forward(self, pixel_values, input_ids, attention_mask):
        with torch.no_grad():
            logits = self.clipseg(pixel_values=pixel_values, input_ids=input_ids, attention_mask=attention_mask).logits
            heatmap = torch.sigmoid(logits).unsqueeze(1)

        s1 = self.cbam1(self.enc1(pixel_values)) 
        s2 = self.cbam2(self.enc2(s1))    
        s3 = self.cbam3(self.enc3(s2))    
        s4 = self.cbam4(self.enc4(s3))    
        b  = self.cbam5(self.enc5(s4))     

        h_b = F.interpolate(heatmap, size=(b.shape[2], b.shape[3]), mode="bilinear", align_corners=True)
        b_fused = b * h_b 
        
        x = self.up4(b_fused) 
        x = torch.cat([x, s4], dim=1); x = self.dec4(x)
        x = self.up3(x); x = torch.cat([x, s3], dim=1); x = self.dec3(x)
        x = self.up2(x); x = torch.cat([x, s2], dim=1); x = self.dec2(x)
        x = self.up1(x); x = torch.cat([x, s1], dim=1); out = self.dec1(x)

        return out

model = CLIPSegForImageSegmentation.from_pretrained("CIDAS/clipseg-rd64-refined").to(device)
processor = CLIPSegProcessor.from_pretrained("CIDAS/clipseg-rd64-refined")
hybrid_model_v1 = VGG19_CLIP_Hybrid_SOTA(model).to(device)

model.safetensors:   0%|          | 0.00/603M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/462 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/380 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/974 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/472 [00:00<?, ?B/s]

/home/bigbro/miniconda3/lib/python3.13/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/bigbro/miniconda3/lib/python3.13/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG19_Weights.IMAGENET1K_V1`. You can also use `weights=VGG19_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/vgg19-dcbb9e9d.pth" to /home/bigbro/.cache/torch/hub/checkpoints/vgg19-dcbb9e9d.pth



  0%|                                                | 0.00/548M [00:00<?, ?B/s]
  0%|                                        | 256k/548M [00:00<05:05, 1.88MB/s]
  0%|                                        | 512k/548M [00:00<05:06, 1.87MB/s]
  0%|                                        | 896k/548M [00:00<03:47, 2.52MB/s]
  0%|                                       | 1.38M/548M [00:00<03:04, 3.11MB/s]
  0%|▏                                      | 2.00M/548M [00:00<02:42, 3.53MB/s]
  0%|▏                                      | 2.50M/548M [00:00<02:26, 3.90MB/s]
  1%|▏                                      | 3.12M/548M [00:00<02:04, 4.58MB/s]
  1%|▎                                      | 3.62M/548M [00:01<02:06, 4.53MB/s]
  1%|▎                                      | 4.25M/548M [00:01<01:54, 5.00MB/s]
  1%|▎                                      | 5.00M/548M [00:01<01:42, 5.53MB/s]
  1%|▍                                      | 5.75M/548M [00:01<01:32, 6.15MB/s]
  1%|▍                     

In [26]:
PATH_HYBRID = "/Models/hybrid_model_v1.pth"
hybrid_model_v1.load_state_dict(torch.load(PATH_HYBRID, map_location=device))
hybrid_model_v1.eval() 

VGG19_CLIP_Hybrid_SOTA(
  (clipseg): CLIPSegForImageSegmentation(
    (clip): CLIPSegModel(
      (text_model): CLIPSegTextModel(
        (embeddings): CLIPSegTextEmbeddings(
          (token_embedding): Embedding(49408, 512)
          (position_embedding): Embedding(77, 512)
        )
        (encoder): CLIPSegEncoder(
          (layers): ModuleList(
            (0-11): 12 x CLIPSegEncoderLayer(
              (self_attn): CLIPSegAttention(
                (k_proj): Linear(in_features=512, out_features=512, bias=True)
                (v_proj): Linear(in_features=512, out_features=512, bias=True)
                (q_proj): Linear(in_features=512, out_features=512, bias=True)
                (out_proj): Linear(in_features=512, out_features=512, bias=True)
              )
              (layer_norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
              (mlp): CLIPSegMLP(
                (activation_fn): QuickGELUActivation()
                (fc1): Linear(in_features=512, out

In [27]:
import os
import cv2
import gradio as gr
import numpy as np
import tensorflow as tf
import torch
import pandas as pd
import traceback
import types
import torch.nn.functional as F
from PIL import Image

def patched_forward(self, pixel_values, input_ids, attention_mask):
    with torch.no_grad():
        logits = self.clipseg(pixel_values=pixel_values, input_ids=input_ids, attention_mask=attention_mask).logits
        heatmap = torch.sigmoid(logits).unsqueeze(1)

    s1 = self.cbam1(self.enc1(pixel_values)) 
    s2 = self.cbam2(self.enc2(s1))    
    s3 = self.cbam3(self.enc3(s2))    
    s4 = self.cbam4(self.enc4(s3))    
    b  = self.cbam5(self.enc5(s4))     

    h_b = F.interpolate(heatmap, size=(b.shape[2], b.shape[3]), mode="bilinear", align_corners=True)
    b_fused = torch.cat([b, h_b], dim=1) 
    
    x = self.up4(b_fused) 
    x = torch.cat([x, s4], dim=1)
    x = self.dec4(x)
    x = self.up3(x)
    x = torch.cat([x, s3], dim=1)
    x = self.dec3(x)
    x = self.up2(x)
    x = torch.cat([x, s2], dim=1)
    x = self.dec2(x)
    x = self.up1(x)
    x = torch.cat([x, s1], dim=1)
    out = self.dec1(x)

    return out

if "hybrid_model_v1" in globals():
    try:
        hybrid_model_v1.forward = types.MethodType(patched_forward, hybrid_model_v1)
        print(" Patch successfully applied on 'hybrid_model_v1'.")
    except Exception as e:
        print(f" Failed to apply channel patch: {e}")

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        tf.config.set_logical_device_configuration(
            gpus[0],
            [tf.config.LogicalDeviceConfiguration(memory_limit=3000)]
        )
        print("GPU: TensorFlow memory limit set to 3000MB.")
    except RuntimeError as e:
        print(f"GPU Config Error: {e}")

device_pytorch = torch.device("cpu")
print("PyTorch / HuggingFace set to: CPU (Thermal Safety Active)\n")

def dice_coef(y_true, y_pred, smooth=1e-5):
    y_true_f = tf.cast(tf.reshape(y_true, [-1]), tf.float32)
    y_pred_f = tf.cast(tf.reshape(y_pred, [-1]), tf.float32)
    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    return (2. * intersection + smooth) / (tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f) + smooth)

def dice_loss(y_true, y_pred): return 1.0 - dice_coef(y_true, y_pred)
def pure_dice_loss(y_true, y_pred): return dice_loss(y_true, y_pred)
def log_cosh_dice_loss(y_true, y_pred):
    x = dice_loss(y_true, y_pred)
    return tf.math.log(tf.math.cosh(x))
def bce_dice_loss(y_true, y_pred):
    bce = tf.keras.losses.binary_crossentropy(y_true, y_pred)
    return bce + dice_loss(y_true, y_pred)
def specificity(y_true, y_pred, smooth=1e-5):
    y_true_f = tf.cast(tf.reshape(y_true, [-1]), tf.float32)
    y_pred_f = tf.cast(tf.reshape(y_pred, [-1]), tf.float32)
    true_negatives = tf.reduce_sum((1.0 - y_true_f) * (1.0 - y_pred_f))
    false_positives = tf.reduce_sum((1.0 - y_true_f) * y_pred_f)
    return (true_negatives + smooth) / (true_negatives + false_positives + smooth)
def recall(y_true, y_pred, smooth=1e-5):
    y_true_f = tf.cast(tf.reshape(y_true, [-1]), tf.float32)
    y_pred_f = tf.cast(tf.reshape(y_pred, [-1]), tf.float32)
    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    return (intersection + smooth) / (tf.reduce_sum(y_true_f) + smooth)
def precision(y_true, y_pred, smooth=1e-5):
    y_true_f = tf.cast(tf.reshape(y_true, [-1]), tf.float32)
    y_pred_f = tf.cast(tf.reshape(y_pred, [-1]), tf.float32)
    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    return (intersection + smooth) / (tf.reduce_sum(y_pred_f) + smooth)

custom_dict = {
    "bce_dice_loss": bce_dice_loss, "pure_dice_loss": pure_dice_loss,
    "log_cosh_dice_loss": log_cosh_dice_loss, "dice_loss": dice_loss,
    "dice_coef": dice_coef, "specificity": specificity,
    "precision": precision, "recall": recall
}
### Change the path if needed
MODEL_DIR = "/Models/"
############################
MODEL_PATHS = {
    "U-Net (BUSI)": os.path.join(MODEL_DIR, "BUSI_UNet.keras"),
    "CBAM-UNet (BUSI)": os.path.join(MODEL_DIR, "BUSI_CBAM_Unet.keras"),
    "VGG19+CBAM (BUSI)": os.path.join(MODEL_DIR, "BUSI_VGG19_trams_Unet_data_aug_FT.keras"),
    "U-Net (DDSM)": os.path.join(MODEL_DIR, "DDSM_UNet.keras"),
    "CBAM-UNet (DDSM)": os.path.join(MODEL_DIR, "DDSM_CBAM_UNet.keras"),
    "VGG19+CBAM (DDSM)": os.path.join(MODEL_DIR, "DDSM_VGG19_U-NET_FT.keras"),
    "ClinicalBERT (DDSM)": os.path.join(MODEL_DIR, "best_multimodal_model_CBERT_FT.keras")
}

MODELS = {}
print("--- Loading Keras Models ---")
for name, path in MODEL_PATHS.items():
    if os.path.exists(path):
        try:
            MODELS[name] = tf.keras.models.load_model(path, custom_objects=custom_dict, compile=False)
            print(f"Loaded : {name}")
        except Exception as e:
            print(f"Error {name} : {e}")
    else:
        print(f"Not Found : {path}")

print("\n--- Loading Bio_ClinicalBERT ---")
from transformers import AutoTokenizer, AutoModel
try:
    clinicalbert_tokenizer = AutoTokenizer.from_pretrained("emilyalsentzer/Bio_ClinicalBERT")
    clinicalbert_model = AutoModel.from_pretrained("emilyalsentzer/Bio_ClinicalBERT").to(device_pytorch)
    print("Bio_ClinicalBERT loaded successfully on CPU.")
except Exception as e:
    print(f"ClinicalBERT Error : {e}")
print("-----------------------------------------\n")

def safe_int(val, default=0):
    if val is None or pd.isna(val) or val in ['n/a', '', 'nan', 'NaN']:
        return default
    try:
        return int(float(val))
    except (ValueError, TypeError):
        return default

def preprocess_image_keras(pil_image, dataset_type, model_choice):
    img = np.array(pil_image.convert('RGB'))
    if dataset_type == "CBIS-DDSM":
        gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
        img = cv2.cvtColor(clahe.apply(gray), cv2.COLOR_GRAY2RGB)
        img = cv2.resize(img, (256, 256))
        img = img / 255.0
        img = (img - np.array([0.485, 0.456, 0.406])) / np.array([0.229, 0.224, 0.225])
    else:
        img = cv2.resize(img, (256, 256))
        if "VGG19" in model_choice:
            img = tf.keras.applications.vgg19.preprocess_input(img.astype(np.float32))
        else:
            img = img / 255.0
    return img

def demo_expert(idx, dataset_choice, model_choice):
    try:
        if idx is None: idx = 0
        idx = int(idx)
        
        if dataset_choice == "BUSI":
            img_array = val_input_imgs[idx].copy()
            if len(img_array.shape) == 2 or img_array.shape[2] == 1:
                img_array = cv2.cvtColor(img_array, cv2.COLOR_GRAY2RGB)
                
            if img_array.max() <= 1.0:
                img_array = (img_array * 255).astype(np.uint8)
            else:
                img_array = img_array.astype(np.uint8)
                
            image_raw = Image.fromarray(img_array)
            h, w = img_array.shape[:2]
            
            gt_bin = val_targets[idx].copy()
            if len(gt_bin.shape) == 3: gt_bin = gt_bin[:, :, 0]
            gt_bin = (gt_bin * 255).astype(np.uint8)
            prompt = "n/a (Unimodal)"
            img_batch = np.expand_dims(val_input_imgs[idx], axis=0)
            
        else: 
            row = val_df.iloc[idx]
            img_path = row['cropped_image_path']
            base_locale = "/home/bigbro/Dataset_CBIS"
            
            if not os.path.exists(img_path):
                if "cbis-ddsm-breast-cancer-image-dataset" in img_path:
                    img_path = os.path.join(base_locale, img_path.split("cbis-ddsm-breast-cancer-image-dataset/")[-1])
                elif "jpeg" in img_path:
                    img_path = os.path.join(base_locale, "jpeg", img_path.split("jpeg/")[-1])
            
            image_raw = Image.open(img_path).convert("RGB")
            img_array = np.array(image_raw)
            h, w = img_array.shape[:2]

            mask_status = row['mask_status']
            if mask_status == 'n/a':
                gt_bin = np.zeros((h, w), dtype=np.uint8)
                prompt = "normal breast tissue"
            else:
                mask_path = row['roi_mask_path']
                if not os.path.exists(mask_path):
                    if "cbis-ddsm-breast-cancer-image-dataset" in mask_path:
                        mask_path = os.path.join(base_locale, mask_path.split("cbis-ddsm-breast-cancer-image-dataset/")[-1])
                    elif "jpeg" in mask_path:
                        mask_path = os.path.join(base_locale, "jpeg", mask_path.split("jpeg/")[-1])
                
                full_mask = Image.open(mask_path).convert("L")
                
                x_min = safe_int(row['x_min'], 0)
                y_min = safe_int(row['y_min'], 0)
                width = safe_int(row['width'], w)
                height = safe_int(row['height'], h)
                
                gt_crop = full_mask.crop((x_min, y_min, x_min + width, y_min + height))
                gt_bin = np.array(gt_crop.resize((w, h)))
                _, gt_bin = cv2.threshold(gt_bin, 127, 255, cv2.THRESH_BINARY)
                
                prompt = f"a {str(row['pathology']).lower()} mass"
            
            img_resized = cv2.resize(img_array, (256, 256))
            processed_img = val_transforms(image=img_resized)["image"]
            
            if torch.is_tensor(processed_img):
                processed_img = processed_img.cpu().numpy()
                if processed_img.shape[0] == 3:
                    processed_img = np.transpose(processed_img, (1, 2, 0))
                    
            img_batch = np.expand_dims(processed_img, axis=0)

        pred_bin = np.zeros((h, w), dtype=np.uint8)
        
        if model_choice == "CLIPSeg (DDSM)":
            if "hybrid_model_v1" in globals() and "processor" in globals():
                hybrid_model_v1.eval().to(device_pytorch)
                inputs = processor(text=[prompt], images=[image_raw], return_tensors="pt", padding="max_length").to(device_pytorch)
                with torch.no_grad():
                    mask_pred = hybrid_model_v1(inputs.pixel_values, inputs.input_ids, inputs.attention_mask)
                    mask_np = mask_pred.squeeze().cpu().numpy()
                
                mask_pred_resized = cv2.resize(mask_np, (w, h), interpolation=cv2.INTER_LINEAR)
                pred_bin = ((mask_pred_resized > 0.5).astype(np.uint8)) * 255
            else:
                msg = "CLIPSeg not instantiated! Please run the notebook cells above to load 'hybrid_model_v1' and 'processor'."
                return img_array, gt_bin, np.zeros((h, w), dtype=np.uint8), img_array, msg
            
        else:
            model = MODELS.get(model_choice)
            if model is not None:
                if "ClinicalBERT" in model_choice:
                    inputs_bert = clinicalbert_tokenizer(prompt, return_tensors="pt", padding=True, truncation=True, max_length=128).to(device_pytorch)
                    with torch.no_grad():
                        outputs_bert = clinicalbert_model(**inputs_bert)
                        text_emb = outputs_bert.last_hidden_state[:, 0, :].cpu().numpy()
                    
                    pred = model.predict([img_batch, text_emb], verbose=0)
                else:
                    pred = model.predict(img_batch, verbose=0)
                
                pred_mask_model = pred[0]
                if len(pred_mask_model.shape) == 3:
                    pred_mask_model = pred_mask_model[:, :, 0]
                
                pred_mask_resized = cv2.resize(pred_mask_model, (w, h), interpolation=cv2.INTER_LINEAR)
                pred_bin = ((pred_mask_resized > 0.5).astype(np.uint8)) * 255

        red_mask = np.zeros_like(img_array)
        red_mask[pred_bin > 0] = [255, 0, 0]
        blended = cv2.addWeighted(red_mask, 0.4, img_array, 1.0, 0)
        
        return img_array, gt_bin, pred_bin, blended, prompt

    except Exception as e:
        print("\nCRASH IN DEMO_EXPERT :")
        traceback.print_exc()
        dummy = np.zeros((256, 256, 3), dtype=np.uint8)
        return dummy, dummy[:,:,0], dummy[:,:,0], dummy, f"Console Error: {e}"

with gr.Blocks(title="PFE: Universal Multimodal Segmentation") as demo:
    gr.Markdown("# Universal Radiology Assistant : Multimodal Segmentation")
    gr.Markdown("USTHB - Compare predictions across all model variants (Keras & PyTorch) on both datasets.")
    
    with gr.Row():
        dataset_select = gr.Radio(choices=["BUSI", "CBIS-DDSM"], label="1. Select Dataset", value="CBIS-DDSM")
        model_select = gr.Dropdown(
            choices=["U-Net (DDSM)", "CBAM-UNet (DDSM)", "VGG19+CBAM (DDSM)", "ClinicalBERT (DDSM)", "CLIPSeg (DDSM)"], 
            label="2. Select Model Variant", 
            value="CLIPSeg (DDSM)"
        )
        
    with gr.Row():
        idx_slider = gr.Slider(0, len(val_df)-1, step=1, label="3. Select Patient Index")
    
    with gr.Row():
        out_img = gr.Image(label="1. Original Image")
        out_gt = gr.Image(label="2. Ground Truth (Radiologist)")
    
    with gr.Row():
        out_pred = gr.Image(label="3. Predicted Mask (AI)")
        out_blend = gr.Image(label="4. Superposition (Overlay)")
        
    out_text = gr.Textbox(label="Clinical Report (Semantic Prior / Diagnosis)")

    def on_dataset_change(dataset):
        if dataset == "BUSI":
            models = ["U-Net (BUSI)", "CBAM-UNet (BUSI)", "VGG19+CBAM (BUSI)"]
            max_val = len(val_input_imgs) - 1
            return gr.Dropdown(choices=models, value=models[0]), gr.Slider(maximum=max_val, value=0)
        else:
            models = ["U-Net (DDSM)", "CBAM-UNet (DDSM)", "VGG19+CBAM (DDSM)", "ClinicalBERT (DDSM)", "CLIPSeg (DDSM)"]
            max_val = len(val_df) - 1
            return gr.Dropdown(choices=models, value="CLIPSeg (DDSM)"), gr.Slider(maximum=max_val, value=0)

    dataset_select.change(on_dataset_change, inputs=dataset_select, outputs=[model_select, idx_slider])

    inputs_list = [idx_slider, dataset_select, model_select]
    outputs_list = [out_img, out_gt, out_pred, out_blend, out_text]
    
    idx_slider.change(fn=demo_expert, inputs=inputs_list, outputs=outputs_list)
    model_select.change(fn=demo_expert, inputs=inputs_list, outputs=outputs_list)

demo.launch(share=True)

✅ Super-Patch de sécurité appliqué sur 'hybrid_model_v1' (Résolution du bug 512/513 canaux).
✅ GPU : TensorFlow limité à 3Go de VRAM.
✅ PyTorch / HuggingFace configuré sur : CPU

--- Chargement des modèles Keras ---


I0000 00:00:1779543966.382802   14071 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 3000 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4060 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9


✅ Chargé : U-Net (BUSI)
✅ Chargé : CBAM-UNet (BUSI)
✅ Chargé : VGG19+CBAM (BUSI)
✅ Chargé : U-Net (DDSM)
✅ Chargé : CBAM-UNet (DDSM)
✅ Chargé : VGG19+CBAM (DDSM)
✅ Chargé : ClinicalBERT (DDSM)

--- Chargement de Bio_ClinicalBERT ---


vocab.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/436M [00:00<?, ?B/s]

...

[transformers] BertModel LOAD REPORT from: emilyalsentzer/Bio_ClinicalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Bio_ClinicalBERT chargé et sécurisé sur CPU.
-----------------------------------------

* Running on local URL:  http://127.0.0.1:7860


model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

* Running on public URL: https://16cce79e0521acf0eb.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


2026-05-23 15:05:38.478320: I external/local_xla/xla/service/service.cc:163] XLA service 0x7833e8002b80 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2026-05-23 15:05:38.478340: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): NVIDIA GeForce RTX 4060 Laptop GPU, Compute Capability 8.9
2026-05-23 15:05:38.496051: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2026-05-23 15:05:38.580866: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91900
I0000 00:00:1779545140.208065   15434 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.
2026-05-23 15:06:37.049342: W external/local_xla/xla/tsl/framework/bfc_allocator.cc:310] Allocator (GPU_0_bfc) ran out of memory trying to allocate 4.08GiB with freed_by_count=0. The c